# ISOT — DistilBERT 256 Token Eğitimi

Yalnızca ISOT kullanılır. Model Colab'a indirilir; WELFake ile eğitilmiş model kullanılmaz.

**Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin. Hücreleri sırayla çalıştırın.

Kod doğrudan GitHub'dan alınır. Hazırlanmış üç ISOT dosyası varsa yeniden temizlenmez. `data/raw` ve `data/ISOT/hazir_v1` konumları kontrol edilir. Dosya bulunamazsa otomatik indirme yapılmaz; aranan yollar gösterilir.

ISOT'ta yayıncı farkı yüksek başarıya yol açabilir; bu deney bir başlangıç ölçümüdür.

In [ ]:
from pathlib import Path
from google.colab import drive
import json
import shutil
import subprocess
import sys

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/fake_news_detection')
PROJECT_DIR = Path('/content/fake_news_detection_isot')
PREPARED_DRIVE = DRIVE_PROJECT / 'data/ISOT/hazir_v1'
RAW_DRIVE = DRIVE_PROJECT / 'data/ISOT/ham'
# Üç hazır dosyayı yüklediğiniz klasör; gerekirse yalnızca bu yolu değiştirin.
UPLOADED_PREPARED_DIR = DRIVE_PROJECT / 'data/raw'
# İlk kez ham veriyle hazırlamak isterseniz açıkça True yapın.
ALLOW_RAW_PREPARATION = False
LOCAL_DATA = Path('/content/isot_hazir_v1')
OUTPUT_ROOT = DRIVE_PROJECT / 'sonuclar/ISOT'
CHECKPOINT_ROOT = DRIVE_PROJECT / 'checkpointler/ISOT'
print('Drive projesi:', DRIVE_PROJECT, flush=True)

def stream_run(command, cwd=None):
    with subprocess.Popen(
        [str(part) for part in command], cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)


## GitHub ve paket kurulumu

Kod GitHub main dalından alınır. Mevcut temiz repo varsa yalnızca ileri yönlü güncellenir; yerel düzenlemeleriniz varsa üzerine yazılmaz. GPU eğitimi henüz başlamaz.

Transformers 5 ve güncel Hugging Face Hub birlikte kurulur. Kurulumdan sonra Hub bağımlılıkları ve eğitim sınıflarının yüklenmesi kontrol edilir. Önceki sürümle hücre çalıştırdıysanız yeni notebook'a geçerken çalışma zamanını yeniden başlatın.

In [ ]:
REPO_URL = 'https://github.com/burak-dursunn/fake-news-detection.git'
if not PROJECT_DIR.exists():
    stream_run(['git', 'clone', '--branch', 'main', REPO_URL, PROJECT_DIR])
else:
    if not (PROJECT_DIR / '.git').is_dir():
        raise RuntimeError(f'{PROJECT_DIR} bir Git reposu değil. PROJECT_DIR için yeni, boş bir klasör adı seçin.')
    origin = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError('Beklenmeyen Git deposu; PROJECT_DIR yolunu kontrol edin.')
    local_changes = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'status', '--porcelain'], text=True).strip()
    if local_changes:
        raise RuntimeError('Colab repo kopyasında yerel değişiklikler var; üzerine yazılmadı. Kurulum için yeni bir PROJECT_DIR seçin veya mevcut değişikliklerinizi saklayın.')
    branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != 'main':
        raise RuntimeError('Colab kopyası main dalında değil; PROJECT_DIR için yeni bir klasör seçin.')
    stream_run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', 'main'])

CODE_SOURCE = {'source': 'github', 'commit': subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True).strip()}
required_code = ['src/prepare_isot.py', 'src/isot_paths.py', 'requirements-colab.txt', 'configs/ISOT_01_DistilBERT_256Token.json']
if not all((PROJECT_DIR / name).is_file() for name in required_code):
    raise RuntimeError('Yeni ISOT kodu GitHub kopyasında bulunamadı; repo ve dalı kontrol edin.')
print('Kullanılan kod:', CODE_SOURCE, flush=True)
stream_run([sys.executable, '-m', 'pip', 'install', '-q', '-r', PROJECT_DIR / 'requirements-colab.txt'])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

environment_check = """
import importlib.metadata as m
from packaging.requirements import Requirement
hub = m.version('huggingface-hub')
for package in ['transformers', 'accelerate', 'gradio', 'gradio-client', 'diffusers']:
    try:
        requirements = m.requires(package) or []
    except m.PackageNotFoundError:
        continue
    for raw in requirements:
        req = Requirement(raw)
        if req.name.replace('_', '-').lower() == 'huggingface-hub':
            if req.marker and not req.marker.evaluate({'extra': ''}):
                continue
            if not req.specifier.contains(hub):
                raise RuntimeError(f'{package}: {req}, installed hub={hub}')
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
import torch
print('Egitim paketleri yuklendi; Hub bagimliliklari uyumlu.')
for package in ['transformers', 'huggingface-hub', 'accelerate', 'torch']:
    print(package, m.version(package))
"""
stream_run([sys.executable, '-c', environment_check], cwd=PROJECT_DIR)


## Yüklenmiş hazır ISOT'u kullan

Önce `UPLOADED_PREPARED_DIR`, ardından standart hazır veri klasörü ve Drive ana dizinindeki `data/raw` kontrol edilir. Tam bir üçlü bulunursa seçilen klasör yazdırılır. Ayrı klasörlerdeki parçalar birleştirilmez. Eksik dosyada indirme/yeniden temizlik yapılmadan durulur.

`ALLOW_RAW_PREPARATION=False` varsayılandır. Hazır veri yoksa ve gerçekten ham dosyadan başlamak istiyorsanız bunu açıkça `True` yapın. Kaynak etiketleri: `True.csv → 0`, `Fake.csv → 1`. `subject` ve `date` modele verilmez.

In [ ]:
from src.prepare_isot import verify_prepared_isot

from src.isot_paths import PREPARED_NAMES, find_prepared_isot

prepared_names = list(PREPARED_NAMES)
candidates = [UPLOADED_PREPARED_DIR, PREPARED_DRIVE, Path('/content/drive/MyDrive/data/raw')]
prepared_files = find_prepared_isot(candidates)
if prepared_files:
    print('Yüklediğiniz hazır ISOT kullanılacak. Yeniden temizlik yapılmayacak.', flush=True)
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    for name, source_path in prepared_files.items():
        print(f'Kopyalanıyor: {source_path}', flush=True)
        shutil.copy2(source_path, LOCAL_DATA / name)
else:
    if not ALLOW_RAW_PREPARATION:
        raise FileNotFoundError(
            'Hazır ISOT bulunamadı; hiçbir dosya indirilmedi. Üç dosyanın olduğu klasörü '
            'UPLOADED_PREPARED_DIR değişkenine yazın. Aranan yollar yukarıda listelendi.'
        )
    RAW_DRIVE.mkdir(parents=True, exist_ok=True)
    if not all((RAW_DRIVE / name).is_file() for name in ['True.csv', 'Fake.csv']):
        import kagglehub
        print('Ham ISOT Kaggle üzerinden indiriliyor...', flush=True)
        dataset_path = Path(kagglehub.dataset_download('rahulogoel/isot-fake-news-dataset'))
        for name in ['True.csv', 'Fake.csv']:
            candidates = [p for p in dataset_path.rglob('*.csv') if p.name.casefold() == name.casefold()]
            if len(candidates) != 1:
                raise RuntimeError(f'{name} için tek bir kaynak dosya bulunamadı.')
            shutil.copy2(candidates[0], RAW_DRIVE / name)
    local_raw = Path('/content/isot_ham')
    local_raw.mkdir(parents=True, exist_ok=True)
    for name in ['True.csv', 'Fake.csv']:
        shutil.copy2(RAW_DRIVE / name, local_raw / name)
    # Yerelde tamamlanmış hazırlık varsa Drive'a kopyalamayı tekrar deneyebiliriz.
    if (LOCAL_DATA / 'isot_hazirlama_raporu.json').is_file():
        verify_prepared_isot(LOCAL_DATA)
    else:
        stream_run([sys.executable, '-u', '-m', 'src.prepare_isot',
                    '--true', local_raw / 'True.csv', '--fake', local_raw / 'Fake.csv',
                    '--output-dir', LOCAL_DATA], cwd=PROJECT_DIR)
    PREPARED_DRIVE.mkdir(parents=True, exist_ok=True)
    for name in prepared_names:
        shutil.copy2(LOCAL_DATA / name, PREPARED_DRIVE / name)
    verify_prepared_isot(PREPARED_DRIVE)
    print("Temiz veri Drive'a kaydedildi:", PREPARED_DRIVE, flush=True)

report = verify_prepared_isot(LOCAL_DATA)
print('Temizleme:', json.dumps(report['cleaning'], ensure_ascii=False, indent=2))
print('Bölmeler:', report['split_counts'])
print('Sınıf sayıları:', report['label_counts_by_split'])
print('Kontroller:', report['audit'])


## Eğitim ve bir kez test

Yeni model `distilbert-base-uncased` temel ağırlıklarından başlar. En iyi model doğrulama F1 skoruyla seçilir. Sonunda test ölçümü ve beş grafik üretilir; sonuçlar doğrudan Drive'a kaydedilir.

Mevcut deney klasörleri korunur. Kesinti olursa otomatik devam yoktur. Test sonucuna göre tekrar tekrar ayar seçmeyin.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı. Çalışma zamanı türünden GPU seçip hücreleri baştan çalıştırın.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
CONFIG = PROJECT_DIR / 'configs/ISOT_01_DistilBERT_256Token.json'
settings = json.loads(CONFIG.read_text(encoding='utf-8'))
assert settings['dataset_name'] == 'isot'
assert settings['model_name'] == 'distilbert-base-uncased'
EXPERIMENT_NAME = settings['experiment_name']
result_dir = OUTPUT_ROOT / EXPERIMENT_NAME
checkpoint_dir = CHECKPOINT_ROOT / EXPERIMENT_NAME
if result_dir.exists() or checkpoint_dir.exists():
    raise FileExistsError(f'Önceki deney korunuyor: {result_dir} veya {checkpoint_dir}. Yeniden çalıştırma için config içinde yeni deney adı kullanın.')

print('Başlatılıyor:', EXPERIMENT_NAME, flush=True)
stream_run([sys.executable, '-u', '-m', 'src.run_experiment',
            '--config', CONFIG, '--data-dir', LOCAL_DATA,
            '--output-root', OUTPUT_ROOT, '--checkpoint-root', CHECKPOINT_ROOT,
            '--skip-split-audit'], cwd=PROJECT_DIR)
(result_dir / 'kod_kaynagi.json').write_text(json.dumps(CODE_SOURCE, indent=2), encoding='utf-8')
print("Sonuçlar ve model Drive'a kaydedildi:", result_dir, flush=True)


## Sonuçları görüntüle

AUC ve AP dört ondalıkla gösterilir; 0,9997 değeri 1,00 gibi görünmez. İki epoch olduğu için doğruluk/F1 eğrisinde yalnızca iki doğrulama noktası bulunması normaldir.

In [ ]:
from IPython.display import Image, display

result_dir = OUTPUT_ROOT / json.loads((PROJECT_DIR / 'configs/ISOT_01_DistilBERT_256Token.json').read_text())['experiment_name']
metrics_file = result_dir / 'test_metrikleri.json'
if not metrics_file.is_file():
    raise FileNotFoundError('Tamamlanmış test sonucu bulunamadı.')
print(json.dumps(json.loads(metrics_file.read_text()), ensure_ascii=False, indent=2))
for path in sorted(result_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))
